In [3]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler

In [4]:
data = pd.read_csv("/content/housing.csv")


In [5]:
X = data[['median_income', 'housing_median_age']]
y = data[['median_house_value']]


In [6]:
scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X)

scaler_y = StandardScaler()
y_scaled = scaler_y.fit_transform(y)


In [7]:
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_tensor = torch.tensor(y_scaled, dtype=torch.float32)


In [8]:
class LinearModel(nn.Module):
    def __init__(self):
        super(LinearModel, self).__init__()
        self.linear = nn.Linear(2, 1)

    def forward(self, x):
        return self.linear(x)

In [9]:
def train_model(reg_type="none", l1_lambda=0.01, l2_lambda=0.01):
    model = LinearModel()
    criterion = nn.MSELoss()
    optimizer = optim.SGD(model.parameters(), lr=0.01)


    print(f"REGULARIZATION TYPE : {reg_type.upper()}")
    print("Weights BEFORE update:", model.linear.weight.data)
    print("Bias BEFORE update   :", model.linear.bias.data)

    # Forward pass
    predictions = model(X_tensor)
    loss = criterion(predictions, y_tensor)

    # Add Regularization
    if reg_type == "l1":
        l1_penalty = sum(torch.sum(torch.abs(p)) for p in model.parameters())
        loss = loss + l1_lambda * l1_penalty

    elif reg_type == "l2":
        l2_penalty = sum(torch.sum(p ** 2) for p in model.parameters())
        loss = loss + l2_lambda * l2_penalty

    elif reg_type == "elastic":
        l1_penalty = sum(torch.sum(torch.abs(p)) for p in model.parameters())
        l2_penalty = sum(torch.sum(p ** 2) for p in model.parameters())
        loss = loss + l1_lambda * l1_penalty + l2_lambda * l2_penalty

    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print("Weights AFTER update :", model.linear.weight.data)
    print("Bias AFTER update    :", model.linear.bias.data)
    print("Loss Value           :", loss.item())


# Execute All Experiments

train_model("none")
train_model("l1")
train_model("l2")
train_model("elastic")

REGULARIZATION TYPE : NONE
Weights BEFORE update: tensor([[ 0.0684, -0.0243]])
Bias BEFORE update   : tensor([-0.4181])
Weights AFTER update : tensor([[ 0.0807, -0.0215]])
Bias AFTER update    : tensor([-0.4098])
Loss Value           : 1.0915604829788208
REGULARIZATION TYPE : L1
Weights BEFORE update: tensor([[0.0110, 0.3193]])
Bias BEFORE update   : tensor([0.1048])
Weights AFTER update : tensor([[0.0252, 0.3150]])
Bias AFTER update    : tensor([0.1026])
Loss Value           : 1.033957839012146
REGULARIZATION TYPE : L2
Weights BEFORE update: tensor([[-0.2233,  0.0258]])
Bias BEFORE update   : tensor([0.2021])
Weights AFTER update : tensor([[-0.2049,  0.0269]])
Bias AFTER update    : tensor([0.1980])
Loss Value           : 1.3954217433929443
REGULARIZATION TYPE : ELASTIC
Weights BEFORE update: tensor([[-0.4721,  0.1866]])
Bias BEFORE update   : tensor([0.5464])
Weights AFTER update : tensor([[-0.4482,  0.1837]])
Bias AFTER update    : tensor([0.5353])
Loss Value           : 2.204990625